# Paper model (ICPhS 2027): cross-language lmer with target-tone grouping, positional informativity

This notebook produces every number and the figure in the paper.

**Model** (cell *lmer fit*): `target_f0 ~ context_f0 * informativity * context_informativity * (unigram + gain) * pos * lang`
+ `context_f0 * duration` + `context_f0 * unit_pos * lang` + target-tone contrasts within language + F0 portion indices, with
random effects `(1 | speaker) + (1 | word_l) + (1 | onset_segment_l) + (1 | offset_segment_l)`
`+ (0 + context_f0 + context_f0:unigram + context_f0:gain || tone_pair_l)` (tone-pair slopes, no tone-pair intercepts)
`+ (1 + context_f0 || target_tone_l)` (surface target tone x language x position).

* No tone-pair intercepts + target-tone grouping: the carryover slope conditions on syllable 2's tone and the anticipatory
  slope on syllable 1's tone, so the two positions are not mirror images of the same covariance.
* Tone-pair slopes: effects on the coarticulation slope (informativity, surprisal, contextual predictability) are tested
  against between-pair slope variation (Satterthwaite df ~ 100-350, not ~500k).
* `informativity` = positional informativity (`tables/informativity_pos_*.csv`): expected surprisal of a tone given the
  adjacent syllable (segments + tone), the direction, and the tone's position in the word. Within-language z-scored;
  context informativity on the target's scale.
* Data: `keep_any2` subset (TSM: syllable 2's surface tone certain; TM: `keep`).

**Order:** settings -> data -> lmer fit (~65-80 min on an Apple M-series laptop; saved and reused) -> contrast cells ->
convergence check -> paper contrasts (`paper_contrasts.R`) -> figure (`paper_figure.R`).
Outputs go to `fits/paper_model/`.


In [ ]:
# install.packages(c("lme4", "lmerTest", "dplyr", "readr", "tidyr", "ggplot2"))
suppressPackageStartupMessages({
  library(lme4); library(lmerTest)
  library(dplyr); library(readr); library(tidyr); library(ggplot2)
})

## Settings

In [ ]:
DATA            <- Sys.getenv("TSM_DATA", path.expand("~/tsm_mfa/all_condenser/processed_data.csv"))   # tsm_pipeline/08_build_processed.py --prep <dir>
COSPRO_DATA     <- Sys.getenv("COSPRO_DATA", "../cospro_pipeline/out/processed_data.csv")   # cospro_pipeline/08_build_processed.py
KEEP_COL        <- "keep_any2"
INF_COL         <- "informativity"   # paper: positional informativity; alternatives: "informativity_full_context", "functional_load_pos"
EXCLUDE_OOV     <- FALSE                    # (Mandarin only; kept for symmetry)
MAX_PER_SPEAKER <- Inf
SEED            <- 2026
FIT_DIR         <- file.path("fits", if (INF_COL == "informativity") "paper_model" else paste0("paper_model_", INF_COL))
dir.create(FIT_DIR, recursive = TRUE, showWarnings = FALSE)
INF_CONTEXT     <- TRUE      # also model the CONTEXT tone's informativity (column context_<INF_COL>)
WORD_RE         <- "intercept"   # word-type random effect: "none" | "intercept" = (1 | word) | "slope" = (1 + context_f0 || word)
WSUF            <- c(none = "", intercept = "_wordint", slope = "_word")[[WORD_RE]]   # suffix for saved fits
add_word <- function(f, g = "word", type = WORD_RE) {
  re <- switch(type, none = NULL, intercept = paste0("(1 | ", g, ")"),
               slope = paste0("(1 + context_f0 || ", g, ")"))
  if (is.null(re)) return(f)
  as.formula(paste(paste(deparse(f, width.cutoff = 500L), collapse = " "), "+", re), env = environment(f))
}
add_terms <- function(f, terms) as.formula(paste(paste(deparse(f, width.cutoff = 500L), collapse = " "), "+", terms),
                                           env = environment(f))

## Data

In [ ]:
TONES <- c("1", "2", "3", "5", "7")
z <- function(x) as.numeric(scale(x))

prep_data <- function(raw, keep_col = KEEP_COL, inf_col = INF_COL, exclude_oov = EXCLUDE_OOV,
                      max_per_speaker = MAX_PER_SPEAKER) {
  d <- raw %>%
    filter(.data[[keep_col]]) %>%
    mutate(
      informativity_raw = .data[[inf_col]],
      context_informativity_raw = if (INF_CONTEXT) .data[[paste0("context_", inf_col)]] else 0,
      target_dur  = if_else(position == "carryover", dur2, dur1),
      position    = factor(position, levels = c("anticipatory", "carryover")),
      pos         = if_else(position == "carryover", 0.5, -0.5),       # effects coding
      medial      = as.numeric(target_pos_class == "medial"),         # target word-medial (control)
      target_tone = factor(as.character(target_tone), levels = TONES),
      tone_pair   = factor(paste0(tone1, "_", tone2, "_", position, "_", target_pos_class, if (INF_CONTEXT) paste0("_", context_pos_class) else "")),
      speaker = factor(speaker), item_id = factor(item_id), word = factor(word),
      onset_segment = factor(onset_segment), offset_segment = factor(offset_segment)
    ) %>%
    filter(!is.na(target_tone), is.finite(target_dur), target_dur > 0,
           is.finite(unigram_surprisal), is.finite(context_gain), is.finite(informativity_raw),
           is.finite(context_informativity_raw))
  if (is.finite(max_per_speaker)) {
    set.seed(SEED)
    d <- d %>% group_by(speaker) %>% slice_sample(n = max_per_speaker) %>% ungroup()
  }
  contrasts(d$target_tone) <- contr.sum(length(TONES))     # tone 7 = negative sum of the other levels
  d
}
scale_vars <- function(d) d %>% mutate(
  unigram         = z(unigram_surprisal),
  gain            = z(context_gain),
  surprisal       = z(word_surprisal),        # only for the comparison model
  informativity   = z(informativity_raw),
  # context informativity on the TARGET's scale, so the two slopes are comparable
  context_informativity = (context_informativity_raw - mean(informativity_raw)) / sd(informativity_raw),
  duration        = z(log(target_dur)),
  target_portion  = z(target_portion),
  context_portion = z(context_portion)
)

raw <- read_csv(DATA, show_col_types = FALSE, guess_max = 100000)
d <- scale_vars(prep_data(raw))

# scaling constants, to translate effects back to bits / raw units
scaling <- tibble(
  variable = c("unigram", "gain", "surprisal", "informativity", "duration"),
  label = c("unigram surprisal (bits)", "contextual gain (bits)", "contextual surprisal (bits)",
            "informativity", "log target duration (s)"),
  mean = c(mean(d$unigram_surprisal), mean(d$context_gain), mean(d$word_surprisal),
           mean(d$informativity_raw), mean(log(d$target_dur))),
  sd   = c(sd(d$unigram_surprisal),   sd(d$context_gain),   sd(d$word_surprisal),
           sd(d$informativity_raw),   sd(log(d$target_dur))))
print(scaling)

cat(nrow(d), "rows |", nlevels(d$speaker), "speakers |", nlevels(d$item_id), "items |",
    n_distinct(d$word), "word types |", nlevels(d$tone_pair), "tone pair x position cells\n")
d %>% count(site) %>% print()
d %>% count(position, target_tone) %>% pivot_wider(names_from = target_tone, values_from = n) %>% print()

In [ ]:
# ---- helper functions used by both the cross-language and the Southern Min-only models ----
tidy_coefs <- function(m) as.data.frame(coef(summary(m))) %>%
  tibble::rownames_to_column("term") %>%
  rename(estimate = Estimate, se = `Std. Error`, t = `t value`, p = `Pr(>|t|)`) %>%
  mutate(l95 = estimate - 1.96 * se, u95 = estimate + 1.96 * se) %>%
  select(term, estimate, se, l95, u95, df, t, p) %>% as_tibble()
fmt <- function(x) x %>% mutate(across(c(estimate, se, l95, u95, t), ~ round(.x, 3)),
                               df = round(df), p = signif(p, 3))
key <- c("context_f0", "context_f0:unigram", "context_f0:gain", "context_f0:surprisal",
         "context_f0:informativity", "context_f0:pos", "context_f0:unigram:pos",
         "context_f0:gain:pos", "context_f0:surprisal:pos", "context_f0:informativity:pos")
rob <- list()
add_rob <- function(name, mm) { rob[[name]] <<- tidy_coefs(mm) %>% mutate(n = nobs(mm)); invisible(NULL) }

slope_ci <- function(m, grid, data = d, factor_col = "target_tone") {
  tt <- delete.response(terms(m, fixed.only = TRUE))
  base <- grid
  if (!is.null(factor_col))
    base[[factor_col]] <- factor(levels(data[[factor_col]])[1], levels = levels(data[[factor_col]]))
  for (v in setdiff(all.vars(tt), c(names(base), "context_f0"))) base[[v]] <- 0   # other predictors at 0
  ca <- if (is.null(factor_col)) NULL else setNames(list(contrasts(data[[factor_col]])), factor_col)
  L  <- model.matrix(tt, mutate(base, context_f0 = 1), contrasts.arg = ca) -
        model.matrix(tt, mutate(base, context_f0 = 0), contrasts.arg = ca)   # slope = L %*% beta
  b  <- fixef(m); V <- as.matrix(vcov(m))
  L  <- L[, names(b), drop = FALSE]
  est <- as.numeric(L %*% b); se <- sqrt(rowSums((L %*% V) * L))
  grid %>% mutate(est = est, l95 = est - 1.96 * se, u95 = est + 1.96 * se)
}


In [ ]:
INF_SCALE <- "within"     # or "pooled"
TM_TONES  <- c("35", "21", "51", "55")

cos_raw <- read_csv(COSPRO_DATA, show_col_types = FALSE, guess_max = 100000)
cos <- cos_raw %>% filter(keep) %>%
  mutate(informativity_raw = .data[[INF_COL]],
         context_informativity_raw = if (INF_CONTEXT) .data[[paste0("context_", INF_COL)]] else 0,
         target_dur = if_else(position == "carryover", dur2, dur1),
         tp = if (grepl("sandhi", INF_COL)) sub("-", "_", surface_pair) else paste0(tone1, "_", tone2),
         unit_prop = phrase_prop) %>%
  filter(as.character(target_tone) %in% TM_TONES, is.finite(target_dur), target_dur > 0,
         is.finite(unigram_surprisal), is.finite(context_gain), is.finite(informativity_raw),
           is.finite(context_informativity_raw))
tsm <- prep_data(raw) %>% mutate(tp = paste0(tone1, "_", tone2), unit_prop = utt_prop)

cols <- c("target_f0", "context_f0", "position", "target_tone", "tp", "speaker", "item_id",
          "onset_segment", "offset_segment", "target_dur", "target_portion", "context_portion",
          "unigram_surprisal", "context_gain", "word_surprisal", "informativity_raw", "context_informativity_raw", "unit_prop", "word", "target_pos_class", "context_pos_class", "sex")
chr <- c("word", "target_tone", "speaker", "item_id", "onset_segment", "offset_segment", "position", "sex")
both <- bind_rows(
  cos %>% select(all_of(cols)) %>% mutate(across(all_of(chr), as.character), lang_lab = "TM"),
  tsm %>% select(all_of(cols)) %>% mutate(across(all_of(chr), as.character), lang_lab = "TSM")) %>%
  mutate(position = factor(position, levels = c("anticipatory", "carryover")),
         pos  = if_else(position == "carryover", 0.5, -0.5),
         lang = if_else(lang_lab == "TSM", 0.5, -0.5),
         tone_pair_l      = factor(paste(lang_lab, tp, position, target_pos_class, if (INF_CONTEXT) context_pos_class else "")),
         target_tone_l    = factor(paste(lang_lab, target_tone, position)),   # SURFACE target tone (target_tone is the surface tone in both languages)
         medial           = as.numeric(target_pos_class == "medial"),
         onset_segment_l  = factor(paste(lang_lab, onset_segment)),
         offset_segment_l = factor(paste(lang_lab, offset_segment)),
         speaker = factor(paste(lang_lab, speaker)),
         word_l  = factor(paste(lang_lab, word))) %>%
  group_by(lang_lab) %>%
  mutate(unigram = z(unigram_surprisal), gain = z(context_gain),
         duration = z(log(target_dur)), target_portion = z(target_portion),
         context_portion = z(context_portion), unit_pos = z(unit_prop),
         informativity = if (INF_SCALE == "within") z(informativity_raw) else informativity_raw,
         # context informativity on the target's scale
         context_informativity = if (INF_SCALE == "within") (context_informativity_raw - mean(informativity_raw)) / sd(informativity_raw)
                                 else context_informativity_raw) %>%
  ungroup()
if (INF_SCALE == "pooled") both <- both %>%
  mutate(context_informativity = (context_informativity - mean(informativity)) / sd(informativity),
         informativity = z(informativity))
# sum-coded target tone within each language, as numeric columns (0 for the other language)
tone_cols <- function(df, lab, levs) {
  cm <- contr.sum(length(levs)); rownames(cm) <- levs
  X <- matrix(0, nrow(df), ncol(cm))
  own <- df$lang_lab == lab
  X[own, ] <- cm[as.character(df$target_tone[own]), , drop = FALSE]
  colnames(X) <- paste0("tone_", lab, seq_len(ncol(X)))
  as_tibble(X)
}
both <- bind_cols(both, tone_cols(both, "TM", TM_TONES), tone_cols(both, "TSM", TONES))
TONE_TERMS <- paste(grep("^tone_(TM|TSM)[0-9]+$", names(both), value = TRUE), collapse = " + ")
stopifnot(!anyNA(select(both, matches("^tone_(TM|TSM)[0-9]+$"))))

both %>% group_by(lang_lab) %>%
  summarise(rows = n(), speakers = n_distinct(speaker), inf_mean = mean(informativity_raw),
            inf_sd = sd(informativity_raw), unigram_mean = mean(unigram_surprisal),
            gain_mean = mean(context_gain), r_gain_position = cor(gain, unit_pos), .groups = "drop") %>% print()
both <- both %>% mutate(sex_c = if_else(sex == "M", 0.5, -0.5))   # effects-coded: F = -0.5, M = +0.5
stopifnot(!anyNA(both$sex_c))

## lmer fit (`m_lang_pos`)

In [ ]:
f_lang <- as.formula(paste(
  "target_f0 ~ context_f0 * informativity * context_informativity * (unigram + gain) * pos * lang + context_f0 * duration + pos * lang +", TONE_TERMS,
  " + target_portion + context_portion +",
  "(1 | onset_segment_l) + (1 | offset_segment_l) + (1 | speaker) +",
  "(0 + context_f0 + context_f0:unigram + context_f0:gain || tone_pair_l) +",   # tone-pair slopes only (underlying tones)
  "(1 + context_f0 || target_tone_l)"))                                           # surface target tone x language x position
# f_lang <- add_terms(f_lang, "context_f0 * medial * ilang")   # word-medial targets
f_lang <- add_word(f_lang, "word_l")   # WORD_RE (Settings)
# f_lang <- add_terms(f_lang, "context_f0 * informativity * (unigram + gain) * sex_c * lang")
f_lang_pos <- add_terms(f_lang, "context_f0 * unit_pos * lang")
f_lang_pos_txt <- paste(deparse(f_lang_pos, width.cutoff = 500L), collapse = " ")
print(f_lang_pos_txt)
# m_lang_pos supplies the formula and the contrast design for brms, and the lmer column of the
# comparison. Reuse a saved fit only if it was fitted with exactly this formula; otherwise refit
# (long: same model as the cross-language cell of analysis_lmer_tsm_citation.ipynb) and save it.
LMER_FILE <- file.path(FIT_DIR, paste0("m_lmer_lang_pos_targettone_", INF_COL, "_", KEEP_COL, "_", INF_SCALE, WSUF, ".rds"))
fit_id    <- paste(f_lang_pos_txt, INF_COL, KEEP_COL, nrow(both), sep = " | ")   # reuse a saved fit only if all match
m_lang_pos <- if (file.exists(LMER_FILE)) readRDS(LMER_FILE) else NULL
if (is.null(m_lang_pos) || !identical(attr(m_lang_pos, "fit_id"), fit_id)) {
  t0 <- Sys.time()
  m_lang_pos <- lmer(f_lang_pos, data = both, REML = TRUE, control = lmerControl(optimizer = "bobyqa"))
  cat("fitted m_lang_pos in", round(difftime(Sys.time(), t0, units = "mins"), 1), "min\n")
  attr(m_lang_pos, "f_txt") <- f_lang_pos_txt
  attr(m_lang_pos, "fit_id") <- fit_id
  saveRDS(m_lang_pos, LMER_FILE)
} else cat("loaded", LMER_FILE, "\n")
tidy_coefs(m_lang_pos) %>% filter(grepl("context_f0", term)) %>% fmt() %>% print(n = Inf)

In [ ]:
# ===================== informativity (target / context) effects on the context-F0 slope =====================
# Marginal effect of X on the coarticulation slope = d²(target_f0) / d(context_f0) d(X), i.e. how much the slope of
# target F0 on context F0 changes per 1 SD of X (X = informativity or context_informativity), evaluated at
#   overall      : lang = 0 (average of the two languages, effects-coded ±0.5), pos = 0 (average of the positions)
#   per language : lang = -0.5 (Mandarin) / +0.5 (Southern Min), pos = 0
#   difference   : Southern Min - Mandarin, pos = 0
# All other continuous predictors at 0 (their means; unigram, gain, unit_pos, duration, portions, sex_c, medial);
# tone terms cancel. Tests: lmerTest::contest with Satterthwaite df.
# BY_POSITION = TRUE adds the anticipatory / carryover rows (x axis of the plots).
library(lmerTest)
MODEL       <- m_lang_pos
BY_POSITION <- TRUE
VARS   <- c(target = "informativity", context = "context_informativity")
HOLD   <- c("unigram", "gain", "informativity", "context_informativity", "duration", "target_portion",
            "context_portion", "unit_pos", "medial", "sex_c")
stopifnot(all(VARS %in% all.vars(formula(MODEL))))
if (!inherits(MODEL, "lmerModLmerTest")) MODEL <- as_lmerModLmerTest(MODEL)

tt_me  <- delete.response(terms(MODEL, fixed.only = TRUE))
ctr_me <- attr(model.matrix(MODEL), "contrasts")
bn_me  <- names(fixef(MODEL))
# contrast row: mixed difference over context_f0 (0/1) and var (0/1) at the given lang and pos
me_L <- function(var, lang_val, pos_val) {
  base <- both[1, ]
  for (v in intersect(HOLD, names(base))) base[[v]] <- 0
  base$lang <- lang_val; base$pos <- pos_val
  X <- function(cf0, x) { nd <- base; nd$context_f0 <- cf0; nd[[var]] <- x
                          model.matrix(tt_me, nd, contrasts.arg = ctr_me)[, bn_me, drop = FALSE] }
  X(1, 1) - X(1, 0) - X(0, 1) + X(0, 0)
}
poss <- if (BY_POSITION) c(average = 0, anticipatory = -0.5, carryover = 0.5) else c(average = 0)
rows <- list()
for (m in names(VARS)) for (p in names(poss)) {
  pv <- poss[[p]]
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "overall",   language = "both languages", L = me_L(VARS[[m]], 0, pv))
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "language",  language = "Mandarin",       L = me_L(VARS[[m]], -0.5, pv))
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "language",  language = "Southern Min",   L = me_L(VARS[[m]], 0.5, pv))
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "difference", language = "Southern Min - Mandarin",
                                   L = me_L(VARS[[m]], 0.5, pv) - me_L(VARS[[m]], -0.5, pv))
}
L_me <- do.call(rbind, lapply(rows, `[[`, "L"))
stopifnot(all(rowSums(abs(L_me)) > 0))

me_tab <- bind_cols(
  tibble(measure  = factor(sapply(rows, `[[`, "measure"), c("target", "context"),
                           c("target informativity", "context informativity")),
         position = factor(sapply(rows, `[[`, "position"), names(poss)),
         type     = sapply(rows, `[[`, "type"),
         language = factor(sapply(rows, `[[`, "language"), c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin"))),
  contest(MODEL, L_me, joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>% as.data.frame() %>%
    transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
  mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
print(as.data.frame(me_tab), row.names = FALSE)
write_csv(me_tab, file.path(FIT_DIR, paste0("informativity_target_context_effects_", INF_SCALE, WSUF, ".csv")))

# sanity check: the average-position difference rows equal the ...:lang coefficients
tidy_coefs(MODEL) %>% filter(term %in% c("context_f0:informativity:lang", "context_f0:lang:context_informativity",
                                         "context_f0:context_informativity:lang")) %>% fmt() %>% print()

# ---- Plots ----
me_plot <- function(tab, ttl, colour_by_language = FALSE) {
  aes_x <- if (BY_POSITION) aes(position, estimate) else aes(language, estimate)
  g <- ggplot(tab, aes_x) + geom_hline(yintercept = 0, linetype = 2, colour = "grey50")
  if (colour_by_language || BY_POSITION) {
    g <- g + geom_pointrange(aes(ymin = l95, ymax = u95, colour = language), position = position_dodge(width = 0.4))
  } else {
    g <- g + geom_pointrange(aes(ymin = l95, ymax = u95))
  }
  g + facet_wrap(~ measure) +
    labs(title = ttl, x = NULL, colour = NULL,
         y = "change in coarticulation slope\nper 1 SD informativity (95% CI)") +
    theme_minimal(base_size = 12) + theme(legend.position = "bottom")
}
p_me_overall <- me_plot(filter(me_tab, type == "overall"),    "Both languages")
p_me_lang    <- me_plot(filter(me_tab, type == "language"),   "By language", colour_by_language = TRUE)
p_me_diff    <- me_plot(filter(me_tab, type == "difference"), "Southern Min - Mandarin")
p_me_overall; p_me_lang; p_me_diff
for (nm in c("overall", "lang", "diff"))
  ggsave(file.path(FIT_DIR, paste0("informativity_target_context_", nm, "_", INF_SCALE, WSUF, ".png")),
         get(paste0("p_me_", nm)), width = 8, height = 4.5, dpi = 300)

In [ ]:
# ===================== informativity x predictability: effect on the unigram / gain effect on the context-F0 slope ======
# For PRED in {unigram, gain} and X in {informativity (target), context_informativity}:
#   d³(target_f0) / d(context_f0) d(PRED) d(X) = how much the PRED effect on the coarticulation slope changes per 1 SD
#   of X (equivalently: how much the X effect on the slope changes per 1 SD of PRED); i.e. context_f0:PRED:X, evaluated
#   overall (lang = 0), per language (lang = ±0.5) and as Southern Min - Mandarin, positions averaged (pos = 0) unless
#   BY_POSITION. Other continuous predictors at 0 (their means); tone terms cancel. lmerTest::contest, Satterthwaite df.
library(lmerTest)
MODEL       <- m_lang_pos
BY_POSITION <- TRUE
VARS  <- c(target = "informativity", context = "context_informativity")
HOLD  <- c("unigram", "gain", "informativity", "context_informativity", "duration", "target_portion",
           "context_portion", "unit_pos", "medial", "sex_c")
if (!inherits(MODEL, "lmerModLmerTest")) MODEL <- as_lmerModLmerTest(MODEL)
tt_ix  <- delete.response(terms(MODEL, fixed.only = TRUE))
ctr_ix <- attr(model.matrix(MODEL), "contrasts")
bn_ix  <- names(fixef(MODEL))

# mixed difference of target_f0 over context_f0 and `vars` (each 0/1), signs (-1)^(k - #ones)
ix_L <- function(vars, lang_val, pos_val) {
  base <- both[1, ]
  for (v in intersect(HOLD, names(base))) base[[v]] <- 0
  base$lang <- lang_val; base$pos <- pos_val
  corners <- expand.grid(rep(list(0:1), length(vars) + 1)); names(corners) <- c("context_f0", vars)
  L <- 0
  for (i in seq_len(nrow(corners))) {
    nd <- base; for (v in names(corners)) nd[[v]] <- corners[[v]][i]
    L <- L + (-1)^(ncol(corners) - sum(corners[i, ])) *
      model.matrix(tt_ix, nd, contrasts.arg = ctr_ix)[, bn_ix, drop = FALSE]
  }
  L
}

interaction_effects <- function(pred) {
  poss <- if (BY_POSITION) c(average = 0, anticipatory = -0.5, carryover = 0.5) else c(average = 0)
  rows <- list()
  for (m in names(VARS)) for (p in names(poss)) {
    v <- c(pred, VARS[[m]]); pv <- poss[[p]]
    rows <- c(rows, list(
      list(measure = m, position = p, type = "overall",    language = "both languages", L = ix_L(v, 0, pv)),
      list(measure = m, position = p, type = "language",   language = "Mandarin",       L = ix_L(v, -0.5, pv)),
      list(measure = m, position = p, type = "language",   language = "Southern Min",   L = ix_L(v, 0.5, pv)),
      list(measure = m, position = p, type = "difference", language = "Southern Min - Mandarin",
           L = ix_L(v, 0.5, pv) - ix_L(v, -0.5, pv))))
  }
  L <- do.call(rbind, lapply(rows, `[[`, "L"))
  miss <- rowSums(abs(L)) == 0
  if (any(miss)) stop("the model has no context_f0:", pred, ":<informativity> term for: ",
                      paste(unique(sapply(rows[miss], `[[`, "measure")), collapse = ", "))
  bind_cols(
    tibble(predictor = pred,
           measure  = factor(sapply(rows, `[[`, "measure"), c("target", "context"),
                             paste0(pred, " x ", c("target informativity", "context informativity"))),
           position = factor(sapply(rows, `[[`, "position"), names(poss)),
           type     = sapply(rows, `[[`, "type"),
           language = factor(sapply(rows, `[[`, "language"),
                             c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin"))),
    contest(MODEL, L, joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>% as.data.frame() %>%
      transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
    mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
}

ix_plot <- function(tab, ttl, colour_by_language = FALSE) {
  aes_x <- if (BY_POSITION) aes(position, estimate) else aes(language, estimate)
  g <- ggplot(tab, aes_x) + geom_hline(yintercept = 0, linetype = 2, colour = "grey50")
  g <- if (colour_by_language || BY_POSITION)
    g + geom_pointrange(aes(ymin = l95, ymax = u95, colour = language), position = position_dodge(width = 0.4))
  else g + geom_pointrange(aes(ymin = l95, ymax = u95))
  g + facet_wrap(~ measure) +
    labs(title = ttl, x = NULL, colour = NULL,
         y = paste0("change in the ", unique(tab$predictor), " effect on the slope\nper 1 SD informativity (95% CI)")) +
    theme_minimal(base_size = 12) + theme(legend.position = "bottom")
}

ix_tabs <- list(); ix_plots <- list()
for (pred in c("unigram", "gain")) {
  tab <- interaction_effects(pred)
  ix_tabs[[pred]] <- tab
  cat("\n=====", pred, "x informativity =====\n"); print(as.data.frame(select(tab, -predictor)), row.names = FALSE)
  ix_plots[[paste0(pred, "_overall")]] <- ix_plot(filter(tab, type == "overall"),    paste(pred, "x informativity: both languages"))
  ix_plots[[paste0(pred, "_lang")]]    <- ix_plot(filter(tab, type == "language"),   paste(pred, "x informativity: by language"), TRUE)
  ix_plots[[paste0(pred, "_diff")]]    <- ix_plot(filter(tab, type == "difference"), paste(pred, "x informativity: Southern Min - Mandarin"))
}
write_csv(bind_rows(ix_tabs), file.path(FIT_DIR, paste0("informativity_x_unigram_gain_effects_", INF_SCALE, WSUF, ".csv")))

# sanity check: the average-position difference rows equal these ...:lang coefficients
tidy_coefs(MODEL) %>% filter(grepl("^context_f0:", term), grepl("lang", term), grepl("informativity", term),
                             grepl("unigram|gain", term), !grepl("pos|unit_pos", term)) %>% fmt() %>% print()

for (nm in names(ix_plots)) {
  print(ix_plots[[nm]])
  ggsave(file.path(FIT_DIR, paste0("informativity_x_", nm, "_", INF_SCALE, WSUF, ".png")),
         ix_plots[[nm]], width = 8, height = 4.5, dpi = 300)
}

In [ ]:
# ===================== target x context informativity =====================
# (a) interaction   : d³(target_f0) / d(context_f0) d(informativity) d(context_informativity) = how much the TARGET-
#                     informativity effect on the coarticulation slope changes per 1 SD of CONTEXT informativity
#                     (= context_f0:informativity:context_informativity), overall / by language / Southern Min - Mandarin
# (b) simple effects: the target-informativity effect on the slope at low / mean / high context informativity
#                     (-1 / 0 / +1 SD), and the context-informativity effect at low / mean / high target informativity
# (c) fitted slopes : coarticulation slope along target informativity (-2..2 SD), one line per context-informativity
#                     level, by language x position, with 95% CIs
# Positions: anticipatory / carryover (POSS). Other continuous predictors at 0 (their means); tone terms cancel.
library(lmerTest)
MODEL <- m_lang_pos
POSS  <- c(anticipatory = -0.5, carryover = 0.5)       # add average = 0 for the position-averaged rows
LANGS <- c(Mandarin = -0.5, "Southern Min" = 0.5)
HOLD  <- c("unigram", "gain", "informativity", "context_informativity", "duration", "target_portion",
           "context_portion", "unit_pos", "medial", "sex_c")
if (!inherits(MODEL, "lmerModLmerTest")) MODEL <- as_lmerModLmerTest(MODEL)
stopifnot("the model has no context_f0 x informativity x context_informativity term" =
            any(grepl("context_f0", names(fixef(MODEL))) & grepl("informativity:.*context_informativity|context_informativity:.*informativity",
                                                                   names(fixef(MODEL)))))
tt_tc  <- delete.response(terms(MODEL, fixed.only = TRUE))
ctr_tc <- attr(model.matrix(MODEL), "contrasts")
bn_tc  <- names(fixef(MODEL))

# mixed difference over context_f0 and `vars` (0/1 corners) with other variables fixed at `at`
L_at <- function(vars, lang_val, pos_val, at = list()) {
  base <- both[1, ]
  for (v in intersect(HOLD, names(base))) base[[v]] <- 0
  base$lang <- lang_val; base$pos <- pos_val
  for (v in names(at)) base[[v]] <- at[[v]]
  corners <- expand.grid(rep(list(0:1), length(vars) + 1)); names(corners) <- c("context_f0", vars)
  L <- 0
  for (i in seq_len(nrow(corners))) {
    nd <- base; for (v in names(corners)) nd[[v]] <- corners[[v]][i]
    L <- L + (-1)^(ncol(corners) - sum(corners[i, ])) *
      model.matrix(tt_tc, nd, contrasts.arg = ctr_tc)[, bn_tc, drop = FALSE]
  }
  L
}
run_contest <- function(L, info) {
  bind_cols(info,
            contest(MODEL, L, joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>% as.data.frame() %>%
              transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
    mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
}
lang_levels <- c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin")

# ---- (a) the interaction ----
V2 <- c("informativity", "context_informativity")
info_a <- list(); L_a <- list()
for (p in names(POSS)) {
  pv <- POSS[[p]]
  for (lg in c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin")) {
    L <- switch(lg, "both languages" = L_at(V2, 0, pv), "Mandarin" = L_at(V2, -0.5, pv),
                "Southern Min" = L_at(V2, 0.5, pv), "Southern Min - Mandarin" = L_at(V2, 0.5, pv) - L_at(V2, -0.5, pv))
    L_a[[length(L_a) + 1]] <- L
    info_a[[length(info_a) + 1]] <- tibble(position = p, language = lg,
                                           type = ifelse(lg == "both languages", "overall", ifelse(grepl(" - ", lg), "difference", "language")))
  }
}
tc_int <- run_contest(do.call(rbind, L_a), bind_rows(info_a)) %>%
  mutate(position = factor(position, names(POSS)), language = factor(language, lang_levels))
cat("\n===== (a) target x context informativity interaction (change in the target-informativity effect on the slope per 1 SD context informativity) =====\n")
print(as.data.frame(tc_int), row.names = FALSE)

# ---- (b) simple effects ----
info_b <- list(); L_b <- list()
for (p in names(POSS)) for (lg in names(LANGS)) for (lev in c(-1, 0, 1)) {
  L_b[[length(L_b) + 1]] <- L_at("informativity", LANGS[[lg]], POSS[[p]], list(context_informativity = lev))
  info_b[[length(info_b) + 1]] <- tibble(effect = "target informativity", at = "context informativity", level = lev, position = p, language = lg)
  L_b[[length(L_b) + 1]] <- L_at("context_informativity", LANGS[[lg]], POSS[[p]], list(informativity = lev))
  info_b[[length(info_b) + 1]] <- tibble(effect = "context informativity", at = "target informativity", level = lev, position = p, language = lg)
}
tc_simple <- run_contest(do.call(rbind, L_b), bind_rows(info_b)) %>%
  mutate(effect = factor(effect, c("target informativity", "context informativity")),
         position = factor(position, names(POSS)),
         level_lab = factor(level, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
cat("\n===== (b) simple effects on the slope (per 1 SD), at low / mean / high levels of the other informativity =====\n")
print(as.data.frame(select(tc_simple, -level_lab)), row.names = FALSE)

# ---- (c) fitted slopes ----
b_tc <- fixef(MODEL); V_tc <- as.matrix(vcov(MODEL))
grid_c <- expand_grid(language = names(LANGS), position = names(POSS), informativity = seq(-2, 2, by = 0.25),
                      context_informativity = c(-1, 0, 1))
L_c <- do.call(rbind, lapply(seq_len(nrow(grid_c)), function(i) with(grid_c[i, ],
          L_at(character(0), LANGS[[language]], POSS[[position]],
               list(informativity = informativity, context_informativity = context_informativity)))))
est <- as.numeric(L_c %*% b_tc); se <- sqrt(rowSums((L_c %*% V_tc) * L_c))
tc_slopes <- grid_c %>% mutate(est = est, l95 = est - 1.96 * se, u95 = est + 1.96 * se,
                               position = factor(position, names(POSS)),
                               context_lab = factor(context_informativity, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))

write_csv(tc_int,    file.path(FIT_DIR, paste0("target_x_context_informativity_interaction_", INF_SCALE, WSUF, ".csv")))
write_csv(tc_simple, file.path(FIT_DIR, paste0("target_x_context_informativity_simple_", INF_SCALE, WSUF, ".csv")))

# sanity check: Southern Min - Mandarin, averaged over positions = the ...:lang coefficient
tidy_coefs(MODEL) %>% filter(grepl("^context_f0:", term), grepl("context_informativity", term),
                             grepl("(^|:)informativity(:|$)", term), grepl("lang", term), !grepl("pos|unigram|gain", term)) %>%
  fmt() %>% print()

# ---- Plots ----
dodge <- position_dodge(width = 0.4)
theme_tc <- theme_minimal(base_size = 12) + theme(legend.position = "bottom")
# 1. the interaction, by position: overall / by language / difference as three panels
p_tc_int <- ggplot(tc_int, aes(position, estimate, colour = language)) +
  geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
  geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
  facet_wrap(~ type, labeller = as_labeller(c(overall = "both languages", language = "by language",
                                              difference = "Southern Min - Mandarin"))) +
  labs(x = NULL, colour = NULL, title = "target x context informativity",
       y = "change in the target-informativity effect on the slope\nper 1 SD context informativity (95% CI)") + theme_tc
# 2. simple effects: each informativity's effect at low / mean / high levels of the other one
p_tc_simple <- ggplot(tc_simple, aes(level_lab, estimate, colour = language)) +
  geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
  geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
  facet_grid(position ~ effect, labeller = labeller(effect = function(x) paste(x, "effect"))) +
  labs(x = "level of the other informativity", colour = NULL,
       y = "change in coarticulation slope per 1 SD (95% CI)") + theme_tc
# 3. fitted coarticulation slopes
p_tc_slopes <- ggplot(tc_slopes, aes(informativity, est, colour = context_lab, fill = context_lab)) +
  geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.12, colour = NA) +
  geom_line(linewidth = 0.9) +
  facet_grid(position ~ language) +
  labs(x = "target informativity (SD)", y = "coarticulation slope (95% CI)",
       colour = "context informativity", fill = "context informativity") + theme_tc
p_tc_int; p_tc_simple; p_tc_slopes
ggsave(file.path(FIT_DIR, paste0("target_x_context_interaction_", INF_SCALE, WSUF, ".png")), p_tc_int, width = 10, height = 4.5, dpi = 300, bg = "white")
ggsave(file.path(FIT_DIR, paste0("target_x_context_simple_", INF_SCALE, WSUF, ".png")), p_tc_simple, width = 9, height = 6.5, dpi = 300, bg = "white")
ggsave(file.path(FIT_DIR, paste0("target_x_context_slopes_", INF_SCALE, WSUF, ".png")), p_tc_slopes, width = 9, height = 6.5, dpi = 300, bg = "white")

In [ ]:
# ===================== target x context informativity: effects on the UNIGRAM / GAIN effect on the slope =============
# Same three analyses as the target x context informativity cell, but for the effect of informativity on
# context_f0:PRED (PRED = unigram or gain), i.e. on how much PRED changes the coarticulation slope:
# (a) interaction   : d⁴(target_f0) / d(context_f0) d(PRED) d(informativity) d(context_informativity)
#                     = context_f0:PRED:informativity:context_informativity (needs that term in the model; skipped if absent)
# (b) simple effects: d³ / d(context_f0) d(PRED) d(X) for X = target (context) informativity at low / mean / high
#                     context (target) informativity
# (c) fitted PRED effect on the slope, d² / d(context_f0) d(PRED), along target informativity, one line per
#                     context-informativity level, by language x position
# Needs the functions and settings of the target x context cell (L_at, run_contest, POSS, LANGS, MODEL, ...).
tc_pred <- function(pred) {
  out <- list()
  V2  <- c(pred, "informativity", "context_informativity")
  # ---- (a) ----
  info_a <- list(); L_a <- list()
  for (p in names(POSS)) {
    pv <- POSS[[p]]
    for (lg in lang_levels) {
      L <- switch(lg, "both languages" = L_at(V2, 0, pv), "Mandarin" = L_at(V2, -0.5, pv),
                  "Southern Min" = L_at(V2, 0.5, pv), "Southern Min - Mandarin" = L_at(V2, 0.5, pv) - L_at(V2, -0.5, pv))
      L_a[[length(L_a) + 1]] <- L
      info_a[[length(info_a) + 1]] <- tibble(position = p, language = lg,
                                             type = ifelse(lg == "both languages", "overall", ifelse(grepl(" - ", lg), "difference", "language")))
    }
  }
  L_a <- do.call(rbind, L_a)
  if (all(L_a == 0)) {
    cat("\n(a) skipped: the model has no context_f0:", pred, ":informativity:context_informativity term\n", sep = "")
    out$int <- NULL
  } else {
    out$int <- run_contest(L_a, bind_rows(info_a)) %>%
      mutate(position = factor(position, names(POSS)), language = factor(language, lang_levels))
    cat("\n=====", pred, "(a): change in the target-informativity effect on the", pred, "effect per 1 SD context informativity =====\n")
    print(as.data.frame(out$int), row.names = FALSE)
  }
  # ---- (b) ----
  info_b <- list(); L_b <- list()
  for (p in names(POSS)) for (lg in names(LANGS)) for (lev in c(-1, 0, 1)) {
    L_b[[length(L_b) + 1]] <- L_at(c(pred, "informativity"), LANGS[[lg]], POSS[[p]], list(context_informativity = lev))
    info_b[[length(info_b) + 1]] <- tibble(effect = "target informativity", level = lev, position = p, language = lg)
    L_b[[length(L_b) + 1]] <- L_at(c(pred, "context_informativity"), LANGS[[lg]], POSS[[p]], list(informativity = lev))
    info_b[[length(info_b) + 1]] <- tibble(effect = "context informativity", level = lev, position = p, language = lg)
  }
  out$simple <- run_contest(do.call(rbind, L_b), bind_rows(info_b)) %>%
    mutate(effect = factor(effect, c("target informativity", "context informativity")),
           position = factor(position, names(POSS)),
           level_lab = factor(level, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
  cat("\n=====", pred, "(b): change in the", pred, "effect on the slope per 1 SD informativity, at low / mean / high levels of the other =====\n")
  print(as.data.frame(select(out$simple, -level_lab)), row.names = FALSE)
  # ---- (c) ----
  L_c <- do.call(rbind, lapply(seq_len(nrow(grid_c)), function(i) with(grid_c[i, ],
            L_at(pred, LANGS[[language]], POSS[[position]],
                 list(informativity = informativity, context_informativity = context_informativity)))))
  est <- as.numeric(L_c %*% b_tc); se <- sqrt(rowSums((L_c %*% V_tc) * L_c))
  out$slopes <- grid_c %>% mutate(est = est, l95 = est - 1.96 * se, u95 = est + 1.96 * se,
                                  position = factor(position, names(POSS)),
                                  context_lab = factor(context_informativity, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
  if (!is.null(out$int)) write_csv(out$int, file.path(FIT_DIR, paste0("target_x_context_", pred, "_interaction_", INF_SCALE, WSUF, ".csv")))
  write_csv(out$simple, file.path(FIT_DIR, paste0("target_x_context_", pred, "_simple_", INF_SCALE, WSUF, ".csv")))

  # ---- plots ----
  if (!is.null(out$int)) {
    out$p_int <- ggplot(out$int, aes(position, estimate, colour = language)) +
      geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
      geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
      facet_wrap(~ type, labeller = as_labeller(c(overall = "both languages", language = "by language",
                                                  difference = "Southern Min - Mandarin"))) +
      labs(x = NULL, colour = NULL, title = paste("target x context informativity, on the", pred, "effect"),
           y = paste0("change in the target-informativity effect on the\n", pred,
                      " effect per 1 SD context informativity (95% CI)")) + theme_tc
  }
  out$p_simple <- ggplot(out$simple, aes(level_lab, estimate, colour = language)) +
    geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
    geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
    facet_grid(position ~ effect, labeller = labeller(effect = function(x) paste(x, "effect"))) +
    labs(x = "level of the other informativity", colour = NULL, title = paste("effects on the", pred, "effect"),
         y = paste0("change in the ", pred, " effect on the slope\nper 1 SD informativity (95% CI)")) + theme_tc
  out$p_slopes <- ggplot(out$slopes, aes(informativity, est, colour = context_lab, fill = context_lab)) +
    geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
    geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.12, colour = NA) +
    geom_line(linewidth = 0.9) +
    facet_grid(position ~ language) +
    labs(x = "target informativity (SD)", y = paste0(pred, " effect on the coarticulation slope\n(per 1 SD ", pred, ", 95% CI)"),
         colour = "context informativity", fill = "context informativity") + theme_tc
  for (nm in c("int", "simple", "slopes")) {
    g <- out[[paste0("p_", nm)]]
    if (is.null(g)) next
    print(g)
    ggsave(file.path(FIT_DIR, paste0("target_x_context_", pred, "_", nm, "_", INF_SCALE, WSUF, ".png")), g,
           width = if (nm == "int") 10 else 9, height = if (nm == "int") 4.5 else 6.5, dpi = 300, bg = "white")
  }
  invisible(out)
}
tc_unigram <- tc_pred("unigram")
tc_gain    <- tc_pred("gain")

## Convergence check, paper contrasts and figure

In [ ]:
# ===================== Convergence check =====================
# lme4 may report "Model failed to converge with max|grad| = 0.005" for this model. Refitting from the converged
# variance parameters leaves REML and all fixed effects unchanged (in our run: max |fixed-effect difference| = 3e-5,
# REML identical to 0.1), i.e. the gradient warning is a false positive of the absolute-gradient check at n > 500k.
RUN_CONV_CHECK <- FALSE   # ~20 min
if (RUN_CONV_CHECK) {
  m2 <- lmer(formula(m_lang_pos), data = model.frame(m_lang_pos), REML = TRUE, start = getME(m_lang_pos, "theta"),
             control = lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))
  print(m2@optinfo$conv$lme4$messages)
  cat("max |fixef diff|:", max(abs(fixef(m2) - fixef(m_lang_pos))), "\n")
  cat("REML:", REMLcrit(m_lang_pos), "->", REMLcrit(m2), "\n")
}


In [ ]:
# ===================== Contrasts reported in the paper =====================
# Every beta / 95% CI / p in the Results section: coarticulation slope (by language x position, language and position
# differences), and the effects of word surprisal (unigram), contextual predictability (gain), target and context
# informativity on the slope. "a" = anticipatory, "c" = carryover; "TSM - TM" and "c - a" are differences.
source("paper_contrasts.R")
paper_tab <- paper_contrasts(m_lang_pos)
options(width = 200); print(paper_tab, row.names = FALSE)
write_csv(paper_tab, file.path(FIT_DIR, "paper_contrasts.csv"))


In [ ]:
# ===================== Figure 1 =====================
source("paper_figure.R")
fig_dat <- paper_figure_data(m_lang_pos, both)
write_csv(fig_dat, file.path(FIT_DIR, "paper_figure.csv"))
paper_figure_plot(fig_dat, file.path(FIT_DIR, "fig1"))
